In [2]:
# ============================================================
# FUCHIBOL26 - Notebook 03: Modelo Poisson
# ============================================================
# Poisson nos permite predecir el MARCADOR EXACTO de un partido
# basándonos en el promedio histórico de goles de cada selección
# ============================================================

import pandas as pd
import os
from math import exp, factorial

# Cargamos los datos
RUTA = os.path.join("..", "datos", "historicos", "resultados.csv")
df = pd.read_csv(RUTA)

# Limpieza rápida
df = df.dropna(subset=["home_score", "away_score"])
df["home_score"] = df["home_score"].astype(int)
df["away_score"] = df["away_score"].astype(int)
df["date"] = pd.to_datetime(df["date"])

# Solo partidos oficiales
TORNEOS_OFICIALES = [
    "FIFA World Cup",
    "FIFA World Cup qualification",
    "UEFA Euro",
    "UEFA Euro qualification",
    "Copa América",
    "African Cup of Nations",
    "African Cup of Nations qualification",
    "AFC Asian Cup",
    "AFC Asian Cup qualification",
    "Gold Cup",
    "UEFA Nations League",
    "CONCACAF Nations League"
]

df_oficial = df[df["tournament"].isin(TORNEOS_OFICIALES)].copy()
df_oficial = df_oficial.sort_values("date").reset_index(drop=True)

# Usamos solo los últimos 8 años — el fútbol cambia mucho
# Lo reciente es más relevante para predecir el futuro
fecha_corte = pd.Timestamp("2018-01-01")
df_reciente = df_oficial[df_oficial["date"] >= fecha_corte].copy()

print("✅ Datos cargados")
print(f"📊 Partidos oficiales totales:  {len(df_oficial):,}")
print(f"📊 Partidos recientes (2018+):  {len(df_reciente):,}")
print(f"📅 Hasta: {df_reciente['date'].max().date()}")

✅ Datos cargados
📊 Partidos oficiales totales:  19,750
📊 Partidos recientes (2018+):  4,788
📅 Hasta: 2026-06-14


In [3]:
# Celda 2: Calcular cuántos goles mete y recibe cada selección en promedio
# Esto es la BASE del modelo Poisson

def calcular_promedios(df, seleccion):
    """
    Calcula el promedio de goles anotados y recibidos
    por una selección en partidos recientes.
    """
    # Partidos como local
    local = df[df["home_team"] == seleccion]
    # Partidos como visitante
    visitante = df[df["away_team"] == seleccion]

    total = len(local) + len(visitante)

    if total == 0:
        return None

    # Goles anotados (a favor)
    goles_anotados = (
        local["home_score"].sum() +
        visitante["away_score"].sum()
    )

    # Goles recibidos (en contra)
    goles_recibidos = (
        local["away_score"].sum() +
        visitante["home_score"].sum()
    )

    return {
        "seleccion":       seleccion,
        "partidos":        total,
        "prom_anotados":   round(goles_anotados / total, 3),
        "prom_recibidos":  round(goles_recibidos / total, 3)
    }

# Calculamos para todas las selecciones
todas = pd.concat([
    df_reciente["home_team"],
    df_reciente["away_team"]
]).unique()

lista = []
for sel in todas:
    p = calcular_promedios(df_reciente, sel)
    if p and p["partidos"] >= 5:  # Mínimo 5 partidos para ser confiable
        lista.append(p)

df_promedios = pd.DataFrame(lista)
df_promedios = df_promedios.sort_values("prom_anotados", ascending=False)
df_promedios = df_promedios.reset_index(drop=True)
df_promedios.index += 1

print("🥅 TOP 15 SELECCIONES — Más goles anotados por partido (2018+)")
print("="*60)
print(df_promedios.head(15).to_string())

# Mostramos selecciones específicas
print("\n🔍 Selecciones de interés:")
for nombre in ["Mexico", "Guatemala", "Brazil", "France", "Spain"]:
    fila = df_promedios[df_promedios["seleccion"] == nombre]
    if len(fila) > 0:
        r = fila.iloc[0]
        print(f"  {nombre:<15} anota: {r['prom_anotados']} | recibe: {r['prom_recibidos']} | partidos: {r['partidos']}")

🥅 TOP 15 SELECCIONES — Más goles anotados por partido (2018+)
             seleccion  partidos  prom_anotados  prom_recibidos
1          New Zealand        11          4.273           0.273
2          Puerto Rico        28          2.714           1.429
3                Japan        57          2.614           0.754
4              Germany        66          2.455           1.152
5              Belgium        79          2.443           0.886
6                Spain        83          2.398           0.843
7          Netherlands        78          2.372           1.000
8              England        84          2.357           0.690
9             Portugal        80          2.300           0.825
10               Haiti        50          2.200           1.180
11                Iran        52          2.192           0.750
12              Canada        63          2.175           0.921
13  Dominican Republic        35          2.171           1.029
14             Algeria        59          

In [4]:
# Celda 3: La fórmula matemática de Poisson
# ============================================================
# Poisson responde: si un equipo mete 2 goles en promedio,
# ¿cuál es la probabilidad de que meta exactamente 0, 1, 2, 3...?
# ============================================================

def poisson(promedio, k):
    """
    Calcula la probabilidad de que ocurran exactamente k goles,
    dado que el promedio esperado es 'promedio'.
    
    Ejemplo:
        poisson(2.0, 0) → probabilidad de meter 0 goles
        poisson(2.0, 1) → probabilidad de meter 1 gol
        poisson(2.0, 2) → probabilidad de meter 2 goles
    """
    return (exp(-promedio) * promedio**k) / factorial(k)

# Ejemplo visual para entender la fórmula
print("📊 EJEMPLO: Si un equipo anota 2.0 goles en promedio...")
print("="*45)
total = 0
for goles in range(7):
    prob = poisson(2.0, goles) * 100
    total += prob
    barra = "█" * int(prob)
    print(f"  {goles} goles → {prob:5.1f}%  {barra}")
print(f"{'='*45}")
print(f"  Total: {total:.1f}%")

print("\n📊 EJEMPLO: Si un equipo anota 1.5 goles en promedio...")
print("="*45)
for goles in range(7):
    prob = poisson(1.5, goles) * 100
    barra = "█" * int(prob)
    print(f"  {goles} goles → {prob:5.1f}%  {barra}")

📊 EJEMPLO: Si un equipo anota 2.0 goles en promedio...
  0 goles →  13.5%  █████████████
  1 goles →  27.1%  ███████████████████████████
  2 goles →  27.1%  ███████████████████████████
  3 goles →  18.0%  ██████████████████
  4 goles →   9.0%  █████████
  5 goles →   3.6%  ███
  6 goles →   1.2%  █
  Total: 99.5%

📊 EJEMPLO: Si un equipo anota 1.5 goles en promedio...
  0 goles →  22.3%  ██████████████████████
  1 goles →  33.5%  █████████████████████████████████
  2 goles →  25.1%  █████████████████████████
  3 goles →  12.6%  ████████████
  4 goles →   4.7%  ████
  5 goles →   1.4%  █
  6 goles →   0.4%  


In [5]:
# Celda 4: Predecir el marcador exacto entre dos selecciones
# Combinamos Poisson de los dos equipos para ver TODOS los marcadores posibles

def predecir_marcador(equipo1, equipo2, max_goles=5):
    """
    Predice la probabilidad de cada marcador posible.
    Devuelve el marcador más probable y las probabilidades
    de victoria, empate y derrota.
    """
    # Buscamos los promedios de cada equipo
    datos1 = df_promedios[df_promedios["seleccion"] == equipo1]
    datos2 = df_promedios[df_promedios["seleccion"] == equipo2]

    if len(datos1) == 0:
        print(f"❌ No encontré datos de '{equipo1}'")
        return
    if len(datos2) == 0:
        print(f"❌ No encontré datos de '{equipo2}'")
        return

    # Promedio de goles anotados y recibidos
    ataque1  = datos1.iloc[0]["prom_anotados"]
    defensa1 = datos1.iloc[0]["prom_recibidos"]
    ataque2  = datos2.iloc[0]["prom_anotados"]
    defensa2 = datos2.iloc[0]["prom_recibidos"]

    # Lambda = ataque del equipo × defensa del rival
    # (cuánto se espera que anote cada equipo)
    lambda1 = (ataque1 + defensa2) / 2
    lambda2 = (ataque2 + defensa1) / 2

    # Calculamos probabilidad de cada marcador posible
    prob_victoria1 = 0  # equipo1 gana
    prob_empate    = 0  # empate
    prob_victoria2 = 0  # equipo2 gana

    mejor_marcador = ""
    mejor_prob     = 0

    print(f"\n{'='*50}")
    print(f"  ⚽ {equipo1}  vs  {equipo2}")
    print(f"{'='*50}")
    print(f"  Goles esperados {equipo1:<12}: {lambda1:.2f}")
    print(f"  Goles esperados {equipo2:<12}: {lambda2:.2f}")
    print(f"\n  📊 Marcadores más probables:")
    print(f"  {'Marcador':<20} {'Prob':>6}")
    print(f"  {'-'*28}")

    resultados = []

    for g1 in range(max_goles + 1):
        for g2 in range(max_goles + 1):
            prob = poisson(lambda1, g1) * poisson(lambda2, g2) * 100
            resultados.append((prob, g1, g2))

            if g1 > g2:
                prob_victoria1 += prob
            elif g1 == g2:
                prob_empate += prob
            else:
                prob_victoria2 += prob

            if prob > mejor_prob:
                mejor_prob = prob
                mejor_marcador = f"{g1} - {g2}"

    # Mostramos los 8 marcadores más probables
    resultados.sort(reverse=True)
    for prob, g1, g2 in resultados[:8]:
        marcador = f"{equipo1[:8]} {g1} - {g2} {equipo2[:8]}"
        print(f"  {marcador:<28} {prob:5.1f}%")

    print(f"\n  {'='*40}")
    print(f"  🏆 Gana {equipo1:<15} {prob_victoria1:5.1f}%")
    print(f"  🤝 Empate              {prob_empate:5.1f}%")
    print(f"  🏆 Gana {equipo2:<15} {prob_victoria2:5.1f}%")
    print(f"  {'='*40}")
    print(f"  ⭐ Marcador más probable: {mejor_marcador} ({mejor_prob:.1f}%)")

# Probamos con partidos interesantes
predecir_marcador("Mexico", "Brazil")
predecir_marcador("Spain", "France")


  ⚽ Mexico  vs  Brazil
  Goles esperados Mexico      : 1.08
  Goles esperados Brazil      : 1.23

  📊 Marcadores más probables:
  Marcador               Prob
  ----------------------------
  Mexico 1 - 1 Brazil           13.2%
  Mexico 0 - 1 Brazil           12.3%
  Mexico 1 - 0 Brazil           10.7%
  Mexico 0 - 0 Brazil           10.0%
  Mexico 1 - 2 Brazil            8.1%
  Mexico 0 - 2 Brazil            7.5%
  Mexico 2 - 1 Brazil            7.1%
  Mexico 2 - 0 Brazil            5.8%

  🏆 Gana Mexico           32.0%
  🤝 Empate               28.2%
  🏆 Gana Brazil           39.5%
  ⭐ Marcador más probable: 1 - 1 (13.2%)

  ⚽ Spain  vs  France
  Goles esperados Spain       : 1.62
  Goles esperados France      : 1.46

  📊 Marcadores más probables:
  Marcador               Prob
  ----------------------------
  Spain 1 - 1 France            10.9%
  Spain 2 - 1 France             8.8%
  Spain 1 - 2 France             7.9%
  Spain 1 - 0 France             7.4%
  Spain 0 - 1 France        

In [6]:
# Celda 5: Predice el partido que quieras
# Cambia los nombres y experimenta

predecir_marcador("Mexico", "Argentina")
predecir_marcador("Guatemala", "United States")
predecir_marcador("Mexico", "Guatemala")


  ⚽ Mexico  vs  Argentina
  Goles esperados Mexico      : 1.14
  Goles esperados Argentina   : 1.17

  📊 Marcadores más probables:
  Marcador               Prob
  ----------------------------
  Mexico 1 - 1 Argentin         13.2%
  Mexico 0 - 1 Argentin         11.6%
  Mexico 1 - 0 Argentin         11.3%
  Mexico 0 - 0 Argentin          9.9%
  Mexico 1 - 2 Argentin          7.7%
  Mexico 2 - 1 Argentin          7.6%
  Mexico 0 - 2 Argentin          6.8%
  Mexico 2 - 0 Argentin          6.5%

  🏆 Gana Mexico           35.1%
  🤝 Empate               28.3%
  🏆 Gana Argentina        36.4%
  ⭐ Marcador más probable: 1 - 1 (13.2%)

  ⚽ Guatemala  vs  United States
  Goles esperados Guatemala   : 1.41
  Goles esperados United States: 1.56

  📊 Marcadores más probables:
  Marcador               Prob
  ----------------------------
  Guatemal 1 - 1 United S       11.3%
  Guatemal 1 - 2 United S        8.8%
  Guatemal 0 - 1 United S        8.0%
  Guatemal 2 - 1 United S        8.0%
  Guatemal 1 

In [7]:
# Celda 6: Guardar los promedios para usar en la simulación del Mundial

RUTA_PROMEDIOS = os.path.join("..", "datos", "procesados", "promedios_goles.csv")
os.makedirs(os.path.dirname(RUTA_PROMEDIOS), exist_ok=True)

df_promedios.to_csv(RUTA_PROMEDIOS, index=False)

print("✅ Promedios guardados")
print(f"📁 Ubicación: datos/procesados/promedios_goles.csv")
print(f"📊 Selecciones guardadas: {len(df_promedios)}")
print(f"\n🏆 Resumen del Modelo Poisson:")
print(f"   Partidos analizados: {len(df_reciente):,} (2018-2026)")
print(f"   Selecciones:         {len(df_promedios)}")
print(f"\n✅ ¡Tienes DOS modelos completos listos!")
print(f"   📈 Modelo Elo     → predice QUIÉN GANA")
print(f"   🎯 Modelo Poisson → predice EL MARCADOR")
print(f"\n🚀 Siguiente paso: Simular el Mundial 2026")

✅ Promedios guardados
📁 Ubicación: datos/procesados/promedios_goles.csv
📊 Selecciones guardadas: 212

🏆 Resumen del Modelo Poisson:
   Partidos analizados: 4,788 (2018-2026)
   Selecciones:         212

✅ ¡Tienes DOS modelos completos listos!
   📈 Modelo Elo     → predice QUIÉN GANA
   🎯 Modelo Poisson → predice EL MARCADOR

🚀 Siguiente paso: Simular el Mundial 2026
